# Solvent recovery skid SR-2 — build the figure and ask the plant

This notebook runs the isometric figure build and then talks to the plant.

It is written for the OpenShell sandbox (`skid`), where the policy allows
GitHub and the deployed figure API. The Gemini key, if you want one, is
injected by an OpenShell provider as `GEMINI_API_KEY` — it is never written
into this file.

In [ ]:
import json
import os
import subprocess
import urllib.request

API = "https://solvent-skid-sr2.lifecyclerd.workers.dev"
REPO = "https://github.com/lifecycleResearch/solvent-skid-sr2"


def sh(cmd, cwd=None):
    print(f"$ {cmd}")
    out = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print(out.stdout[-4000:], out.stderr[-2000:])
    return out


def get_json(url):
    with urllib.request.urlopen(url, timeout=30) as r:
        return json.load(r)

## 1. Clone and build the figure

In [ ]:
sh(f"rm -rf /sandbox/solvent-skid-sr2 && git clone --depth 1 {REPO} /sandbox/solvent-skid-sr2")
sh("node figures/skid/build.mjs && node figures/skid/build.mjs --light", cwd="/sandbox/solvent-skid-sr2")
sh("node scripts/lines.mjs figures/skid/skid-light.html", cwd="/sandbox/solvent-skid-sr2")

## 2. Read the plant's description from the deployed API

In [ ]:
plant = get_json(f"{API}/api/context")
print(plant["name"], "—", plant["subtitle"])
print()
print(plant["description"])
print()
print("equipment:", len(plant["equipment"]), "| beds:", len(plant["beds"]), "| vacuum sets:", len(plant["vacuum"]))

In [ ]:
for q in ["what does the 13X bed take", "pure solvent vs pure product", "vacuum stages and boiling points"]:
    answer = get_json(f"{API}/api/describe?q={urllib.parse.quote(q)}")["answer"]
    print(f"--- {q}\n{answer}\n")

## 3. Ask a model, with the key the sandbox was given

In [ ]:
key = os.environ.get("GEMINI_API_KEY", "")
print("key present:", bool(key))

if key:
    body = json.dumps({
        "provider": "gemini",
        "model": "gemini-2.5-flash",
        "key": key,
        "state": {"mode": "product", "stages": 3, "cuts": [3]},
        "messages": [{"role": "user", "content": "Walk me through the pure product setup with a three stage vacuum."}],
    }).encode()
    req = urllib.request.Request(f"{API}/api/chat", data=body, headers={"content-type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as r:
        text = ""
        for raw in r:
            line = raw.decode().strip()
            if not line.startswith("data:"):
                continue
            payload = line[5:].strip()
            if not payload:
                continue
            obj = json.loads(payload)
            text += obj.get("text", "")
    print(text)
else:
    print("No GEMINI_API_KEY in this sandbox. Attach a provider:")
    print("  openshell provider create --name google --type google --credential GEMINI_API_KEY=$GEMINI_API_KEY")
    print("  openshell sandbox provider attach skid --provider google")

## 4. The geometry audit

In [ ]:
sh("node figures/skid/build.mjs --audit", cwd="/sandbox/solvent-skid-sr2")
audit = get_json(f"{API}/api/context")["audit"]
print(json.dumps(audit, indent=2))